<a href="https://colab.research.google.com/github/redfelix/CustomGLNotebook/blob/main/TrascrizioneV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
## FASE 1 - Installazione dipendenze
!pip install -U openai-whisper
!sudo apt update
##sudo apt install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 41.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=96db813885edde323e944d1a562cb5d30aead6acb098bded99cade2ff5b479c3
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,878 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Ge

In [3]:
import os
# import subprocess
import torch
import whisper
from google.colab import files


# Modelli: tiny, base, small, medium, large, large-v3
modello = "large-v3"  # @param ["tiny", "base", "small", "medium", "large", "large-v3"]
lingua = "it"  # @param ["it", "en", "auto"]
elimina_file_sorgente = True  # @param {type:"boolean"}
scarica_txt = True  # @param {type:"boolean"}

# Tempo di inizio e fine (formato HH:MM:SS oppure MM:SS, lasciare 0 per tutto il file)
start_time = "00:00:00"  # @param {type:"string"}
end_time = "00:00:00"    # @param {type:"string"}


# Metodo per convertire HH:MM:SS o MM:SS in secondi per clip_timestamps
def converti_in_secondi(time_str):
    time_str = str(time_str).strip()
    if not time_str or time_str == "0":
        return 0.0
    parti = time_str.split(":")
    if len(parti) == 3:
        return int(parti[0]) * 3600 + int(parti[1]) * 60 + float(parti[2])
    elif len(parti) == 2:
        return int(parti[0]) * 60 + float(parti[1])
    else:
        return float(time_str)

# Controllo se disponibile la GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo in uso: {device.upper()}")
if device == "cpu":
    print("Warn: nessuna GPU attiva. Per velocizzare seleziona una GPU da Runtime > Modifica tipo di runtime e seleziona T4.")

print(f"Caricamento modello '{modello}' in corso...")
model = whisper.load_model(modello, device=device)
print("Modello caricato.")

# Caricamento file audio/video
print("\nCarica il file audio o video:")
uploaded = files.upload()

if not uploaded:
    print("Nessun file selezionato.")
else:
    for file_name in uploaded.keys():
        print(f"\n--- Inizio elaborazione: {file_name} ---")

        '''
        # gestione del file temporaneo con ffmpeg
        input_audio = file_name
        temp_clipped = None
        has_start = start_time.strip() not in ["", "0", "00:00:00", "00:00"]
        has_end = end_time.strip() not in ["", "0", "00:00:00", "00:00"]
        if has_start or has_end:
            temp_clipped = f"temp_cut_{os.path.splitext(file_name)[0]}.wav"
            cmd = ["ffmpeg", "-y", "-loglevel", "error"]
            if has_start:
                cmd.extend(["-ss", start_time])
            cmd.extend(["-i", file_name])
            if has_end:
                cmd.extend(["-to", end_time])
            cmd.extend(["-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", temp_clipped])
            print(f"Estrazione intervallo audio ({start_time} -> {end_time if has_end else 'fine'})...")
            subprocess.run(cmd, check=True)
            input_audio = temp_clipped
        '''

        # Trascrizione del testo
        print("Trascrizione in corso con Whisper...")
        opzioni = {"verbose": False}
        if lingua != "auto":
            opzioni["language"] = lingua

        '''
        clip_timestamps: Union[str, List[float]]
            Comma-separated list start,end,start,end,... timestamps (in seconds) of clips to process.
            The last end timestamp defaults to the end of the file.

        parser.add_argument("--clip_timestamps", type=str, default="0", help="comma-separated list start,end,start,end,... timestamps (in seconds) of clips to process, where the last end timestamp defaults to the end of the file")
        '''

        # Configurazione intervallo clip direttamente con le opzioni native di transcribe
        sec_start = converti_in_secondi(start_time)
        sec_end = converti_in_secondi(end_time)

        if sec_start > 0 or sec_end > 0:
            if sec_end > 0:
                opzioni["clip_timestamps"] = [sec_start, sec_end]
            else:
                opzioni["clip_timestamps"] = [sec_start]

            print(f"Intervallo impostato nativamente: {sec_start}s -> {sec_end if sec_end > 0 else 'fine'}")

        ''' trascrizione sul file tagliato esternamente
        # result = model.transcribe(input_audio, **opzioni)
        '''

        # Trascrizione diretta sul file originale usando clip_timestamps
        result = model.transcribe(file_name, **opzioni)

        # Mostra il testo completo
        print("\n" + "=" * 50)
        print("TESTO TRASCRITTO:")
        print("=" * 50)
        print(result["text"].strip())
        print("=" * 50)

        # Dettaglio con i minuti dei segmenti
        print("\nSegmenti:")
        for seg in result["segments"]:
            m_in, s_in = divmod(seg["start"], 60)
            m_out, s_out = divmod(seg["end"], 60)
            print(f"[{int(m_in):02d}:{s_in:05.2f} -> {int(m_out):02d}:{s_out:05.2f}] {seg['text'].strip()}")

        # Salvataggio e clean
        nome_base = os.path.splitext(file_name)[0]
        txt_path = f"{nome_base}_trascrizione.txt"

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(result["text"].strip())

        if scarica_txt:
            print(f"\nDownload del file di testo: {txt_path}")
            files.download(txt_path)

        '''
        # pulizia del file temporaneo tagliato da ffmpeg
        # if temp_clipped and os.path.exists(temp_clipped):
        #     os.remove(temp_clipped)
        '''

        if elimina_file_sorgente and os.path.exists(file_name):
            os.remove(file_name)
            print(f"File sorgente {file_name} rimosso.")

        print(f"\nTrascrizione-elaborazione terminata per {file_name}.")

Dispositivo in uso: CUDA
Caricamento modello 'large-v3' in corso...


100%|██████████████████████████████████████| 2.88G/2.88G [00:13<00:00, 228MiB/s]


Modello caricato.

Carica il file audio o video:


Saving 2025-11-27 14-28-18.mp4 to 2025-11-27 14-28-18.mp4

--- Inizio elaborazione: 2025-11-27 14-28-18.mp4 ---
Trascrizione in corso con Whisper...


100%|██████████| 942/942 [00:01<00:00, 505.74frames/s]


TESTO TRASCRITTO:
Grazie.

Segmenti:
[00:00.00 -> 00:02.00] Grazie.

Download del file di testo: 2025-11-27 14-28-18_trascrizione.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File sorgente 2025-11-27 14-28-18.mp4 rimosso.

Trascrizione-elaborazione terminata per 2025-11-27 14-28-18.mp4.
